# Habitat suitability under climate change

Our changing climate is changing where plant species can live,
and conservation and restoration practices will need to take
this into
account.

In this coding challenge, you will create a habitat suitability model
for a terrestrial plant species of your choice that lives in the contiguous United States
(CONUS). We have this limitation because the downscaled climate data we
suggest, the [MACAv2 dataset](https://www.climatologylab.org/maca.html),
is only available in the CONUS – if you find other downscaled climate
data at an appropriate resolution, you are welcome to choose a different
study area. If you don’t have anything in mind, you can take a look at
[*Sorghastrum nutans*](https://www.gbif.org/species/2704414), a grass native to North America. In the past 50
years, its range has moved
northward.

Your suitability assessment will be based on combining multiple data
layers related to soil, topography, and climate, then applying a fuzzy logic model across the different data layers to generate habitat suitability maps. 

You will need to create a **modular, reproducible, workflow** using functions and loops.
To do this effectively, we recommend planning your code out in advance
using a technique such as a pseudocode outline or a flow diagram. We
recommend breaking each of the blocks below out into multiple steps. It
is unnecessary to write a step for every line of code unless you find
that useful. As a rule of thumb, aim for steps that cover the major
structures of your code in 2-5 line chunks.

## Load Libraries


In [ ]:
## Load Libraries

# Reproducable file paths
import os
from glob import glob
import pathlib 
from pathlib import Path

# GBIF packages
import pygbif.occurrences as occ
import pygbif.species as species
from getpass import getpass

# Unzipping
import zipfile
import time

# Spatial Data
import geopandas as gpd
import xrspatial
import xarray as xr
from rioxarray.merge import merge_arrays
import rioxarray.merge as rxrm
import rioxarray as rxr

# Other Data Types
import numpy as np
import pandas as pd
from math import floor, ceil

## Invalid Geometries 
from shapely.geometry import MultiPolygon, Polygon

## Visualizations
import holoviews as hv
import hvplot.pandas
import hvplot.xarray
import matplotlib.pyplot as plt

# For API use
import requests

## Set Directories

In [ ]:
## Project Directory
data_dir = os.path.join(

    pathlib.Path.home(),
    'Earth Data Analytics',
    'Spring 26',
    'habitat',
    'suit_final'
)

os.makedirs(data_dir, exist_ok=True)

In [ ]:
## Site Directories

site_dir_CO = Path(data_dir) / "site_aspen_CO"
site_dir_CO.mkdir(parents = True, exist_ok=True)

site_dir_UT = Path(data_dir) / "site_aspen_UT"
site_dir_UT.mkdir(parents = True, exist_ok=True)

# Protected Area Datasets [Colorado and Utah]
# http://www.sciencebase.gov/catalog/item

In [ ]:
## GBIF Data Directory

gbif_dir = os.path.join(data_dir, 'gbif_aspen')

In [ ]:
## Soil Properties Directories 

# Soil pH Rasters
ph_co_raster_dir = os.path.join(site_dir_CO, "soil pH", "Gunnison", "rasters")
os.makedirs(ph_co_raster_dir, exist_ok = True)

ph_ut_raster_dir = os.path.join(site_dir_UT, "soil pH", "Fishlake", "rasters")
os.makedirs(ph_ut_raster_dir, exist_ok = True)

# Soil Organic Matter Rasters
om_co_raster_dir = os.path.join(site_dir_CO, "soil Organic Matter", "Gunnison", "rasters")
os.makedirs(om_co_raster_dir, exist_ok = True)

om_ut_raster_dir = os.path.join(site_dir_UT, "soil Organic Matter", "Fishlake", "rasters")
os.makedirs(om_ut_raster_dir, exist_ok = True)

# Soil pH Plots
ph_co_plots_dir = os.path.join(site_dir_CO, "soil pH", "Gunnison", "plots")
os.makedirs(ph_co_plots_dir, exist_ok = True)

ph_ut_plots_dir = os.path.join(site_dir_UT, "soil pH", "Fishlake", "plots")
os.makedirs(ph_ut_plots_dir, exist_ok = True)

# Soil Organic Matter Plots
om_co_plots_dir = os.path.join(site_dir_CO, "soil Organic Matter", "Gunnison", "plots")
os.makedirs(om_co_plots_dir, exist_ok = True)

om_ut_plots_dir = os.path.join(site_dir_UT, "soil Organic Matter", "Fishlake", "plots")
os.makedirs(om_ut_plots_dir, exist_ok = True)

In [ ]:
### Topographic Data Directories

## Make data directory for topography
elev_dir = os.path.join(data_dir, "topography")
os.makedirs(elev_dir, exist_ok = True)

## Make subdirectory for the Gunnison Forest Data
gun_topo_dir = os.path.join(elev_dir, "gun")
os.makedirs(gun_topo_dir, exist_ok = True)

## Make subdirectory for the Fishlake National Forest Data
fish_topo_dir = os.path.join(elev_dir, "fish")
os.makedirs(fish_topo_dir, exist_ok = True)

In [ ]:
## Climate Data Directory
maca_dir = os.path.join(data_dir, 'maca-dir')
os.makedirs(maca_dir, exist_ok = True)

maca_pattern = os.path.join(maca_dir, '*.nc')
maca_pattern

## Collect GBIF Data

In [ ]:
## Reset credentials
reset_credentials = False

credentials = dict(
    GBIF_USER=(input, 'GBIF username:'),
    GBIF_PWD=(input, 'GBIF password:'),
    GBIF_EMAIL=(input, 'GBIF email:')
)

## Loop through credentials and enter them
for env_variable, (prompt_func, prompt_text) in credentials.items():


    if reset_credentials and (env_variable in os.environ):
        os.environ.pop(env_variable)

    if not env_variable in os.environ:
        os.environ[env_variable] = prompt_func(prompt_text)

In [ ]:
## Species Name
species_name = "Populus tremuloides"

## Species info from GBIF
species_info = species.name_lookup(species_name, 
                                   rank = 'SPECIES')
## Grab first result
first_result = species_info['results'][0]
first_result

In [ ]:
## Get species key
species_key = first_result['nubKey']

## Check
first_result['species'], species_key

In [ ]:
## Assign species code
species_key = 3040215

In [ ]:
## Make file path
gbif_pattern = os.path.join(gbif_dir, '*.csv')

## Download it once
if not glob(gbif_pattern):

    ## Submit query
    gbif_query = occ.download([
        f"speciesKey = {species_key}",
        "hasCoordinate = True",
    ]) 

    ## Only download once
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]
        download_key = os.environ['GBIF_DOWNLOAD_KEY']

        ## Wait for the download to build
        wait = occ.download_meta(download_key)['status']
        while not wait == 'SUCCEEDED':
            wait = occ.download_meta(download_key)['status']
            time.sleep(5)

    ## Download data
    download_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'],
        path = data_dir
    )

    ## Unzip the file
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path = gbif_dir)

    # Find csv file path
gbif_path = glob(gbif_pattern)[0]
gbif_path

In [ ]:
gbif_df = pd.read_csv(
    gbif_path,
    delimiter = '\t'
)

gbif_df.head()

## Plot GBIF Data of Aspens in the US

In [ ]:
# Make it spatial (geodataframe)
gbif_gdf = (
    gpd.GeoDataFrame(
        gbif_df,
        geometry = gpd.points_from_xy(
            gbif_df.decimalLongitude,
            gbif_df.decimalLatitude,
        ),
        crs = 'EPSG: 4326'
    )
)

In [ ]:
## Plot it
gbif_gdf.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'American Aspen Occurrences in GBIF',
    fill_color = None,
    line_color = 'orange',
    framewidth = 600
)

In [ ]:
# Define location of the data
ITEM_ID_CO = "6759abcfd34edfeb8710a004"
FILENAME_CO = "PADUS4_1_State_CO_GDB_KMZ.zip"

ITEM_ID_UT = "6759abcfd34edfeb8710a004"
FILENAME_UT = "PADUS4_1_State_UT_GDB_KMZ.zip"


## Make URL
url_co = f"https://www.sciencebase.gov/catalog/file/get/{ITEM_ID_CO}?name={FILENAME_CO}?"
url_ut = f"https://www.sciencebase.gov/catalog/file/get/{ITEM_ID_UT}?name={FILENAME_UT}?"

## Place for the shapefile to live
output_path_co = site_dir_CO / FILENAME_CO
output_path_ut = site_dir_UT / FILENAME_UT

In [ ]:
## Collect data for National Parks boundaries 

## API Call
with requests.get(url_co, stream = True) as r:
    r.raise_for_status()
    with open(output_path_co, "wb") as f:
        for chunk in r.iter_content(chunk_size = 8192):
            if chunk:
                f.write(chunk)


## API Call
with requests.get(url_ut, stream = True) as r:
    r.raise_for_status()
    with open(output_path_ut, "wb") as f:
        for chunk in r.iter_content(chunk_size = 8192):
            if chunk:
                f.write(chunk)

In [ ]:
## Unzip Colorado

## Place for unzipped files to live
zip_path_co = Path(output_path_co)

## Folder for the data 
extract_folder_co = zip_path_co.parent

## Create the folder if it doesn't exist
extract_folder_co.mkdir(parents = True, exist_ok = True)

## Unzip
with zipfile.ZipFile(zip_path_co, 'r') as zip_ref:
    zip_ref.extractall(extract_folder_co)

In [ ]:
## Unzip Utah

## Place for unzipped files to live
zip_path_ut = Path(output_path_ut)

## Folder for the data 
extract_folder_ut = zip_path_ut.parent

## Create the folder if it doesn't exist
extract_folder_ut.mkdir(parents = True, exist_ok = True)

## Unzip
with zipfile.ZipFile(zip_path_ut, 'r') as zip_ref:
    zip_ref.extractall(extract_folder_ut)

In [ ]:
import fiona

In [ ]:
pa_path_co = extract_folder_co / "PADUS4_1_StateCO.gdb"

## List layers
layers_co = fiona.listlayers(pa_path_co)
layers_co

In [ ]:
## Open fee layers
pa_shp_co = gpd.read_file(pa_path_co, layer = "PADUS4_1Fee_State_CO")
pa_shp_co

In [ ]:
pa_path_ut = extract_folder_ut / "PADUS4_1_StateUT.gdb"

## List layers
layers_ut = fiona.listlayers(pa_path_ut)
layers_ut

In [ ]:
## Open fee layers
pa_shp_ut = gpd.read_file(pa_path_ut, layer = "PADUS4_1Fee_State_UT")
pa_shp_ut

In [ ]:
pa_shp_co.crs

In [ ]:
## Change CRS to 4326 for plotting
pa_shp_co = pa_shp_co.to_crs(epsg = 4326)
pa_shp_ut = pa_shp_ut.to_crs(epsg = 4326)

### Plot National Park Boundaries

In [ ]:
pa_shp_co.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Colorado Protected Areas',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

In [ ]:
pa_shp_ut.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Utah Protected Areas',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

In [ ]:
pa_shp_co.columns

### Intersect State Map with GBIF Data

In [ ]:
## Intersect with GBIF data
aspen_co = gpd.overlay(gbif_gdf, pa_shp_co, how = 'intersection')

In [ ]:
## Sum the number of occurrences per site
value_counts_co = aspen_co['Loc_Nm'].value_counts()
value_counts_co

In [ ]:
## Intersect with GBIF data
aspen_ut = gpd.overlay(gbif_gdf, pa_shp_ut, how = 'intersection')

In [ ]:
## Sum the number of occurrences per site
value_counts_ut = aspen_ut['Loc_Nm'].value_counts()
value_counts_ut

In [ ]:
pd.reset_option('display.max_rows')

### Subset to National Parks with GBIF Data

In [ ]:
## Subset to Gunnison National Forest
gun_gdf = pa_shp_co[pa_shp_co['Unit_Nm'] == 'Gunnison National Forest']
gun_gdf

In [ ]:
## Subset to Fishlake National Forest
fish_gdf = pa_shp_ut[pa_shp_ut['Unit_Nm'] == 'Fishlake National Forest']
fish_gdf

In [ ]:
gun_gdf.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Gunnison National Forest',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

In [ ]:
fish_gdf.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Fishlake National Forest',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

In [ ]:
## Combine into a single GDF
sites_gdf = gpd.GeoDataFrame(pd.concat([gun_gdf, fish_gdf], ignore_index = True))
sites_gdf

In [ ]:
sites_gdf.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Gunnison and Fishlake National Forest',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

In [ ]:
## Make a bounding box for my study site for pH soil (Gunnison National Forest)

xmin, ymin, xmax, ymax = gun_gdf.total_bounds

## Initialize tiles to accumulate into
tiles_gun = []

## Loop through lat/lon to create grid tiles
for lat_min in range(floor(ymin), ceil(ymax)):
    for lon_min in range(floor(xmin), ceil(xmax)):

        ## Calculate max lat and lon for tile
        lat_max, lon_max = lat_min + 1, lon_min + 1

        ## url for pH data
        ph_url_gun = ("http://hydrology.cee.duke.edu/POLARIS/PROPERTIES/v1.0/ph/mean/15_30/"
        f"/lat{lat_min}{lat_max}_lon{lon_min}{lon_max}.tif")

         ## Open raster
        raster = rxr.open_rasterio(ph_url_gun)

        ## CLEAN THE RASTER HERE
        raster = raster.where((raster > 0) & (raster < 1000))  # remove invalid values
     

        ## Open raster and append to tiles list
        tiles_gun.append(raster)



In [ ]:
## Make a bounding box for my study site for pH soil (Fishlake National Forest)

xmin, ymin, xmax, ymax = fish_gdf.total_bounds

## Initialize tiles to accumulate into
tiles_fish = []

## Loop through lat/lon to create grid tiles
for lat_min in range(floor(ymin), ceil(ymax)):
    for lon_min in range(floor(xmin), ceil(xmax)):

        ## Calculate max lat and lon for tile
        lat_max, lon_max = lat_min + 1, lon_min + 1

        ## url for pH data
        ph_url_fish = ("http://hydrology.cee.duke.edu/POLARIS/PROPERTIES/v1.0/ph/mean/15_30/"
        f"/lat{lat_min}{lat_max}_lon{lon_min}{lon_max}.tif")

         ## Open raster
        raster = rxr.open_rasterio(ph_url_fish)

        ## CLEAN THE RASTER HERE
        raster = raster.where((raster > 0) & (raster < 1000))  # remove invalid values
     

        ## Open raster and append to tiles list
        tiles_fish.append(raster)



In [ ]:
## Merge all the individual ph tiles
gun_ph_da = rxrm.merge_arrays(tiles_gun).rio.clip_box(*gun_gdf.total_bounds)

## Merge all the individual ph tiles
fish_ph_da = rxrm.merge_arrays(tiles_fish).rio.clip_box(*fish_gdf.total_bounds)

gun_om_da = gun_ph_da

fish_om_da = fish_ph_da



In [ ]:
gun_ph_da.plot()

In [ ]:
fish_ph_da.plot()

In [ ]:
## Make a function to get the urls for the soil data from POLARIS
def create_polaris_urls(soil_prop, stat, soil_depth, gdf_bounds):

  """
  Function to generate dataset of POLARIS urls using site boundary
     
  Args: 
  soil_prop (str): soil property that we want (pH, organic matter, etc)
  stat (str): summary statistic (mean, p5, etc.)
  soil_depth (str): soil depth in cm 
  gdf_bounds: array of site boundaries
    
  Returns:
  list: a list of POLARIS urls  
  
  """
    
  ## Extract bounding box for the site
  min_lon, min_lat, max_lon, max_lat = gdf_bounds
    
  ## Snap boundaries to whole degrees
  site_min_lon = floor(min_lon)
  site_min_lat = floor(min_lat)
  site_max_lon = ceil(max_lon)
  site_max_lat = ceil(max_lat)

  # Initialize output list
  all_soil_urls = []

  ## Loop through lat and lon to get each tile in study area
  for lon in range(site_min_lon, site_max_lon):
    for lat in range(site_min_lat, site_max_lat):

      ## Define the corners of the tile
      current_max_lon = lon + 1
      current_max_lat = lat + 1

      ## Define the url template
      soil_template = (
        "http://hydrology.cee.duke.edu/POLARIS/PROPERTIES/v1.0/"

        ## Placeholders for parameters we want to vary
        "{soil_prop}/"
        "{stat}/"
        "{soil_depth}/"
        "/lat{min_lat}{max_lat}_lon{min_lon}{max_lon}.tif"
      )
          
      ## Fill in the template with the parameters for one complete URL
      soil_url = soil_template.format(
        soil_prop = soil_prop,
        stat = stat,
        soil_depth = soil_depth,
        min_lat = lat, max_lat = current_max_lat,
        min_lon = lon, max_lon = current_max_lon
      )
        
      ## Add the url to the list
      all_soil_urls.append(soil_url)
        
  return all_soil_urls

In [ ]:
## Function to open the raster tiles, mask and scale them, clip them to the site, and merge them
def build_da(urls, bounds):

    """
    Build a dataarray from list of urls.

    Args:
    urls (list): list of urls where the data live
    bounds (tuple): site boundaries

    Returns:
    xarray.DataArray: merged DataArray
    """

    ## Initialize an empty list
    all_das = []

    ## Add a buffer to keep all data
    buffer = 0.025
    xmin, ymin, xmax, ymax = bounds
    bounds_buffer = (xmin - buffer, ymin - buffer, xmax + buffer, ymax + buffer)

    ## Process 1 url tile at a time
    for url in urls:

        ## Open raster, mask missing data, remove any extra dimensions
        tile_da = rxr.open_rasterio(url,
                                    mask_and_scale = True).squeeze() 
        
        tile_da = tile_da.where((tile_da > 0) & (tile_da < 1000))

        ## Unpack bounds and crop the tile to buffered boundaries
        cropped_da = tile_da.rio.clip_box(*bounds_buffer)

        ## Store cropped tile
        all_das.append(cropped_da)

    ## Outside loop - Combine into a single raster
    merged = merge_arrays(all_das)

    ## Return the final raster
    return merged

In [ ]:
## Function to save an xarray.DataArray as a raster
def export_raster(da, raster_path, data_dir):

    """
    Export raster to file
    
    Args:
    raster (xarray.DataArray): input raster layer
    raster_path (str): output raster directory
    data_dir (str): path of data directory
    
    Returns: None, save to disk instead
    """

    output_file = os.path.join(data_dir, os.path.basename(raster_path))
    da.rio.to_raster(output_file)

In [ ]:
## Function for customizable plots
def plot_site(site_da, site_gdf, plots_dir, site_fig_name, plot_title, 
              bar_label, plot_cmap, boundary_clr, tif_file = False):
    
    """
    Create a custom site plot
    
    Args:
    site_da (xarray.DataArray): input site raster
    site_gdf (geopandas.GeoDataFrame): site boundary gdf
    plots_dir (str): path of plots directory for saving plots
    site_fig_name (str): site figure name
    plot_title (str): plot title
    bar_label (str): plot bar variable name
    plot_cmap (str): colormap for the plot
    boundary_clr (str): color for site boundary
    tif_file (bool): indicate if there is a site file to draw from
    
    Returns:
    matplotlib.pyplot.plot: a plot of site values
    """

    ## Set up the figure
    fig = plt.figure(figsize = (8,6))
    ax = plt.axes()

    ## Conditional
    if tif_file:

        # If we've already made a plot, just open it
        site_da = rxr.open_rasterio(site_da, masked = True) 

    # Otherwise make plot of dataarray values
    site_plot = site_da.plot(cmap = plot_cmap,
                             cbar_kwargs = {'label': bar_label})
    
    ## Plot site boundary
    site_gdf.boundary.plot(ax = plt.gca(), color = boundary_clr)

    plt.title(f'{plot_title}')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

    fig.savefig(f"{plots_dir}/{site_fig_name}.png")

    return site_plot

In [ ]:
# Create wrapper function
def download_polaris(site_name, site_gdf, soil_prop, stat, soil_depth,
                     plot_path, plot_title, data_dir, plots_dir):
    
    """
    Retrieve POLARIS data, build DataArray, plot site, and export raster
    
    Args:
    site_name (str): Name of the site, used to name exported raster file
    site_gdf (geopandas.GeoDataFrame): boundary of site, used for bounding box
    soil_prop (str): Soil property of interest
    stat (str): Summary statistic for POLARIS data
    soil_depth (str): Soil depth of focus in CM
    plot_path (str): Used to build plot filename
    plot_title (str): text for title of plot
    data_dir (str): Path for the data directory where rasters will be saved
    plots_dir (str): Path for plots directory where PNG plot files will be saved
    
    Returns:
    xarray.DataArray: soil DataArray for given location
    """

    ## Collect the soil URLS
    site_polaris_urls = create_polaris_urls(soil_prop, stat, soil_depth, site_gdf.total_bounds)

    ## Download rasters, gather into single file
    site_soil_da = build_da(site_polaris_urls, tuple(site_gdf.total_bounds))

    ## Export as a raster
    export_raster(site_soil_da, f"{site_name}_soil_{soil_prop}.tif", data_dir)

    ## Plot site
    plot_site(site_soil_da, site_gdf, plots_dir,
              f'{plot_path}-Soil', f'{plot_title=}',
              soil_prop, 'viridis', 'white')
    
    ## Return the soil raster
    return site_soil_da

In [ ]:
## Get soil data array for Gunnison National Forest
gun_soil_ph_da = download_polaris(site_name = "Gunnison",
                               site_gdf = gun_gdf,
                               soil_prop = "ph",
                               stat = "mean",
                               soil_depth = "15_30",
                               plot_path = "ph_15_30_gunnison",
                               plot_title = "Soil pH in Gunnison National Forest",
                               data_dir = ph_co_raster_dir,
                               plots_dir = ph_co_plots_dir
                               )

gun_soil_om_da = download_polaris(site_name = "Gunnison",
                               site_gdf = gun_gdf,
                               soil_prop = "om",
                               stat = "mean",
                               soil_depth = "15_30",
                               plot_path = "ph_15_30_gunnison",
                               plot_title = "Soil % Organic Matter in Gunnison National Forest",
                               data_dir = om_co_raster_dir,
                               plots_dir = om_co_plots_dir
                               )

In [ ]:
## Get Soil Data Array for Fishlake National Forest

fish_soil_ph_da = download_polaris(site_name = "Fishlake",
                               site_gdf = fish_gdf,
                               soil_prop = "ph",
                               stat = "mean",
                               soil_depth = "15_30",
                               plot_path = "ph_15_30_fishlake",
                               plot_title = "Soil pH in Fishlake National Forest",
                               data_dir = ph_ut_raster_dir,
                               plots_dir = ph_ut_plots_dir
                               )

fish_soil_om_da = download_polaris(site_name = "Fishlake",
                               site_gdf = fish_gdf,
                               soil_prop = "om",
                               stat = "mean",
                               soil_depth = "15_30",
                               plot_path = "ph_15_30_fishlake",
                               plot_title = "Soil % Organic Matter in Fishlake National Forest",
                               data_dir = om_ut_raster_dir,
                               plots_dir = om_ut_plots_dir
                               )

## Collect Topographic Data

In [ ]:
import earthaccess

In [ ]:
## Set up Earth Access
earthaccess.login()

In [ ]:
## Sesrch for SRTM data
datasets = earthaccess.search_datasets(keyword = "SRTM DEM")
for dataset in datasets:
    print(dataset['umm']['ShortName'], dataset['umm']['EntryTitle'])

### Collect Topography Data for Gunnison National Forest

In [ ]:
## File pattern for data
gun_srtm_pattern = os.path.join(gun_topo_dir, '*.hgt.zip')

## Study area for topo data
gun_elev_bounds = tuple(gun_gdf.total_bounds)

## Add buffer
buffer = 0.025
gun_xmin, gun_ymin, gun_xmax, gun_ymax, = gun_elev_bounds
gun_elev_bounds_buffer = (gun_xmin - buffer,
                          gun_ymin - buffer,
                          gun_xmax + buffer,
                          gun_ymax + buffer)

## Look at results
if not glob(gun_srtm_pattern):

    ## Search for Data
    gun_srtm_search = earthaccess.search_data(
        short_name = 'SRTMGL1',
        bounding_box = gun_elev_bounds_buffer
    )

    ## Download data
    gun_srtm_results = earthaccess.download(
        gun_srtm_search,
        gun_topo_dir
    )

else:
    print("SRTM files already downloaded.")

In [ ]:
### Visualize the topographic data 

gun_srtm_da_list = []
for srtm_path in glob(gun_srtm_pattern):
    tile_da = rxr.open_rasterio(srtm_path, mask_and_scale = True).squeeze()
    srtm_cropped_da = tile_da.rio.clip_box(*gun_elev_bounds_buffer)
    gun_srtm_da_list.append(srtm_cropped_da)

## Merge
gun_srtm_da = merge_arrays(gun_srtm_da_list)

gun_srtm_da.plot(cmap='terrain')

gun_gdf.boundary.plot(ax = plt.gca(), color='black')

In [ ]:
# Reproject to projected CRS
gun_rpj = gun_srtm_da.rio.reproject("EPSG:5070")

## Get aspect layer from elevation layer
gun_aspect = xrspatial.aspect(gun_srtm_da)
gun_northness = np.cos(np.deg2rad(gun_aspect))

## Plot it
gun_northness.plot(cmap="RdBu")

gun_gdf.boundary.plot(ax = plt.gca(), 
            edgecolor = 'black',
            )

## Cut off values less than 0

### Collect Topography Data for Fishlake National Forest

In [ ]:
## Collect Topographic Data for Fishlake National Forest

## File pattern for data
fish_srtm_pattern = os.path.join(fish_topo_dir, '*.hgt.zip')

## Study area for topo data
fish_elev_bounds = tuple(fish_gdf.total_bounds)

## Add buffer
buffer = 0.025
fish_xmin, fish_ymin, fish_xmax, fish_ymax, = fish_elev_bounds
fish_elev_bounds_buffer = (fish_xmin - buffer,
                          fish_ymin - buffer,
                          fish_xmax + buffer,
                          fish_ymax + buffer)

## Look at results
if not glob(fish_srtm_pattern):

    ## Search for Data
    fish_srtm_search = earthaccess.search_data(
        short_name = 'SRTMGL1',
        bounding_box = fish_elev_bounds_buffer
    )

    ## Download data
    fish_srtm_results = earthaccess.download(
        fish_srtm_search,
        fish_topo_dir
    )

else:
    print("SRTM files already downloaded.")



In [ ]:
### Visualize the topographic data 

fish_srtm_da_list = []
for srtm_path in glob(fish_srtm_pattern):
    tile_da = rxr.open_rasterio(srtm_path, mask_and_scale = True).squeeze()
    srtm_cropped_da = tile_da.rio.clip_box(*fish_elev_bounds_buffer)
    fish_srtm_da_list.append(srtm_cropped_da)

## Merge
fish_srtm_da = merge_arrays(fish_srtm_da_list)

fish_srtm_da.plot(cmap='terrain')

fish_gdf.boundary.plot(ax = plt.gca(), color='black')

In [ ]:
# Reproject to projected CRS
fish_rpj = fish_srtm_da.rio.reproject("EPSG:5070")

## Get aspect layer from elevation layer
fish_aspect = xrspatial.aspect(fish_srtm_da)

fish_aspect = fish_aspect.where(fish_aspect >= 0)

## Plot it
fish_aspect.plot(cmap = 'terrain')

fish_gdf.boundary.plot(ax = plt.gca(), 
            edgecolor = 'black',
            )

## Cut off values less than 0

## Collect Climate Model Data

In [ ]:
## Temperature in Kelvin 
def convert_temperature(temp):
    """
    A function to convert temperature from Kelvin to Fahrenheit

    Args:
    temp (int): Temperature listed in Kelvin

    Results:
    temp_f (int): Temperature listed in Fahrenheit
    """

    return temp * 1.8 - 459.67

In [ ]:
## Convert longitude
def convert_longitude(longitude):

    return (longitude - 360) if longitude > 180 else longitude

In [ ]:
def create_maca_url(model, rcp_value, date_range):

    maca_url = (
    "http://thredds.northwestknowledge.net:8080/thredds/dodsC"
    "/MACAV2"
    f"/{model}"
    "/macav2metdata_pr" # {whatever variable you want to use}, pr, tasmin, etc.
    f"_{model}_r1i1p1"
    f"_{rcp_value}" # choose emission scenario
    f"_{date_range}_CONUS"
    "_monthly.nc"
    )

    return maca_url

In [ ]:
## Function to process and download MACA data
def process_maca_da(site_dict,
                    years_list,
                    models_list,
                    rcp_value,
                    maca_data_dir):
    results = []
    ## Loop over both sites in site_dict
    for site_name, site_gdf in site_dict.items():

        ## Loop over each time period
        for date_range in years_list:

            ## Loop over each climate model
            for model in models_list:
                print(f"Processing {site_name} | {model} | {date_range}")
                
                ## Define the MACA URL
                maca_url = create_maca_url(model, rcp_value, date_range)

                ## Make MACA path
                maca_path = os.path.join(
                    maca_data_dir,
                    f"maca_{model}_{site_name}_{rcp_value}_{date_range}_CONUS_monthly.nc"
                )

                ## Download the data only once
                if not os.path.exists(maca_path):

                    ## Open remote data set and squeeze the data
                    maca_da = xr.open_dataset(maca_url).squeeze().precipitation

                    ## Save locally
                    maca_da.to_netcdf(maca_path)

                    print("Downloaded", {maca_path}, "Successfully")

                else: 
                    print("File", {maca_path}, "already exists.")
                    maca_da = xr.open_dataset(maca_path).precipitation


                ## Define the spatial bounds
                bounds_maca = site_gdf.total_bounds

                ## Change longitude value to match gdf
                maca_da = maca_da.assign_coords(
                    lon=("lon", [convert_longitude(x) for x in maca_da.lon.values])
                )

                ## Set spatial dimensions
                maca_da = maca_da.rio.set_spatial_dims(
                    x_dim="lon",
                    y_dim="lat"
                )

                ## Crop to site boundaries
                maca_da_cropped = maca_da.rio.clip_box(*bounds_maca)

                

                ## Add cropped da to a dictionary with metadata and save to list
                result = dict(
                    site_name = site_name,
                    climate_model = model,
                    date_range = date_range,
                    da = maca_da_cropped
                )
                
                results.append(result)
    
    #Return list of dictionaries, each with cropped and processed climate data
    return results

 

In [ ]:
all_maca_data = process_maca_da(site_dict = {"gun": gun_gdf, "fish": fish_gdf},
                                years_list = ["2031_2035", "2036_2040", "2041_2045", "2046_2050", "2051_2055", "2056_2060"],
                                models_list = ["CanESM2","GFDL-ESM2M", "HadGEM2-ES365", "MRI-CGCM3"],
                                rcp_value = "rcp45",
                                maca_data_dir = maca_dir)

## Harmonize the Data

In [ ]:
### Align the grids of the different data layers

## Make a list of the layers
gun_das_list = [
    gun_soil_ph_da,
    gun_srtm_da,
    gun_northness,
    gun_soil_om_da   
]

fish_das_list = [
    fish_soil_ph_da,
    fish_srtm_da,
    fish_aspect,
    fish_soil_om_da   
]

In [ ]:
## Define the boundaries
gun_bounds = tuple(gun_gdf.total_bounds)

## Add a small buffer
buffer = 0.025
(gun_xmin, gun_ymin, gun_xmax, gun_ymax) = gun_bounds

## Buffer bounding box
gun_bounds_buffer = (gun_xmin - buffer,
                          gun_ymin - buffer,
                          gun_xmax + buffer,
                          gun_ymax + buffer)

In [ ]:
## Define the boundaries
fish_bounds = tuple(fish_gdf.total_bounds)

## Add a small buffer
buffer = 0.025
(fish_xmin, fish_ymin, fish_xmax, fish_ymax) = fish_bounds

## Buffer bounding box
fish_bounds_buffer = (fish_xmin - buffer,
                          fish_ymin - buffer,
                          fish_xmax + buffer,
                          fish_ymax + buffer)

In [ ]:
## Check out pre-cropped boundaries
print(gun_ph_da.rio.bounds())
print(gun_srtm_da.rio.bounds())
print(gun_northness.rio.bounds())
print(gun_soil_om_da.rio.bounds())

In [ ]:
## Check out pre-cropped boundaries
print(fish_ph_da.rio.bounds())
print(fish_srtm_da.rio.bounds())
print(fish_aspect.rio.bounds())
print(fish_soil_om_da.rio.bounds())

In [ ]:
## Add a name to the arrays for 1 site
gun_soil_ph_da.name = "Gunnison Soil pH"
gun_srtm_da.name = "Gunnison Elevation"
gun_aspect.name = "Gunnison Aspect"
gun_northness.name = "Gunnison Northness"
gun_soil_om_da.name = "Gunnison Organic Matter"

## Add a name to the arrays for 1 site
fish_soil_ph_da.name = "Fishlake Soil pH"
fish_srtm_da.name = "Fishlake Elevation"
fish_aspect.name = "Fishlake Aspect"
fish_soil_om_da.name = "Fishlake Organic Matter"

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
## Empty list for cropped and reprojected DAs
reproj_da_list_gun = []
reproj_da_list_fish = []

## Loop through Gunnison DAs
for da in tqdm(gun_das_list):

    ## If Gunnison is in the name, do this
    if 'Gunnison' in da.name:

        ## crop the DA
        cropped_da = da.rio.clip_box(*gun_bounds_buffer)

        ## Reproject and match
        reproj_da_gun = (cropped_da.rio.reproject_match(gun_ph_da))

        ## Add it to the list
        reproj_da_list_gun.append(reproj_da_gun)

reproj_da_list_gun

## Loop through Fishlake DAs
for da in tqdm(fish_das_list):

    ## If Fishlake is in the name, do this
    if 'Fishlake' in da.name:

        ## crop the DA
        cropped_da = da.rio.clip_box(*fish_bounds_buffer)

        ## Reproject and match
        reproj_da_fish = (cropped_da.rio.reproject_match(fish_ph_da))

        ## Add it to the list
        reproj_da_list_fish.append(reproj_da_fish)

reproj_da_list_fish

In [ ]:
print(gun_ph_da.rio.bounds())
print(reproj_da_list_gun[2].rio.bounds())

print(fish_ph_da.rio.bounds())
print(reproj_da_list_fish[1].rio.bounds())



In [ ]:
## Create subplots for Gunnison National Forest
fig, axes = plt.subplots(1, len(reproj_da_list_gun),
                         figsize = (5*len(reproj_da_list_gun), 5))

if len(reproj_da_list_gun) == 1:
    axes = [axes]

for ax, data in zip(axes, reproj_da_list_gun):
    if data.ndim == 3:
        data = data.squeeze()

    data.plot(ax = ax, cmap = 'viridis', add_colorbar = True)

    gun_gdf.plot(ax = ax, facecolor = 'none', edgecolor = 'white', linewidth = 1)

    ax.set_aspect("equal")
    ax.set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
## Create subplots for Fishlake National Forest
fig, axes = plt.subplots(1, len(reproj_da_list_fish),
                         figsize = (5*len(reproj_da_list_fish), 5))

if len(reproj_da_list_fish) == 1:
    axes = [axes]

for ax, data in zip(axes, reproj_da_list_fish):
    if data.ndim == 3:
        data = data.squeeze()

    data.plot(ax = ax, cmap = 'viridis', add_colorbar = True)

    fish_gdf.plot(ax = ax, facecolor = 'none', edgecolor = 'white', linewidth = 1)

    ax.set_aspect("equal")
    ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
def harmonize_raster_layers_gun(reference_raster_gun, input_rasters, data_dir):

    """
    Harmonize raster layers to ensure consisten spatial resolution and projection
    
    Args:
    reference_raster (str): Path of raster to reference.
    input_rasters (list): List of rasters to harmonize.
    data_dir (str): Path of data directory

    Returns:
    list: A list of harmonized rasters
    """

    harmonized_files = []

    harmonized_files.append(reference_raster_gun)

    ## Load the reference raster
    ref_raster = rxr.open_rasterio(reference_raster_gun, masked = True)

    ## Use projection EPSG:4326
    ref_raster = ref_raster.rio.write_crs(4326)

    for raster_path in input_rasters:

        # Load the input raster
        input_raster = rxr.open_rasterio(raster_path, masked = True)
        input_raster = input_raster.rio.write_crs(4326)

        ## Reproject and align the input raster to match the reference raster

        # Only 2D/3D arrays with dimnesions x/y are currently supported
        # by reproject_match()
        harmonized_raster = input_raster.rio.reproject_match(ref_raster)

        #Save the harmonized raster to the output directory
        output_file = os.path.join(data_dir, os.path.basename(raster_path))
        harmonized_raster.rio.to_raster(output_file)
        harmonized_files.append(output_file)

    return harmonized_files

In [ ]:
## Reference Raster for Gunnison and Fishlake National Forest
reference_raster_gun = f"{data_dir}/site_aspen_CO/soil pH/rasters/Gunnison_soil_ph.tif"

reference_raster_fish = f"{data_dir}/site_aspen_UT/soil pH/rasters/Fishlake_soil_ph.tif"

print(reference_raster_gun)
print(reference_raster_fish)

In [ ]:
import os
import glob
import xarray as xr
import rioxarray as rxr

def convert_maca_nc_to_tif(
    nc_dir,
    out_dir,
    variable="precipitation",
    summary_method="mean",   # "mean", "sum", or None
    clip_bounds=None         # e.g. gun_gdf.total_bounds
):
    """
    Convert a folder of MACA NetCDF files to GeoTIFF rasters.

    Parameters
    ----------
    nc_dir : str, Folder containing .nc files
    out_dir : str, Folder to save .tif files
    variable : str, Variable name in the NetCDF
    summary_method : str or None
        How to summarize over time: "mean", "sum", or None
    clip_bounds : tuple or None
        Bounding box as (minx, miny, maxx, maxy)

    Returns
    -------
    list
        List of output GeoTIFF file paths
    """
    os.makedirs(out_dir, exist_ok=True)

    nc_files = glob.glob(os.path.join(nc_dir, "*.nc"))
    tif_files = []

    for fp in nc_files:
        print(f"Processing {os.path.basename(fp)}")

        ds = xr.open_dataset(fp)
        da = ds[variable]

        # Remove length-1 dimensions if present
        da = da.squeeze()

        # Summarize time dimension if present
        if "time" in da.dims and summary_method is not None:
            if summary_method == "mean":
                da = da.mean(dim="time")
            elif summary_method == "sum":
                da = da.sum(dim="time")
            else:
                raise ValueError("summary_method must be 'mean', 'sum', or None")

        # Rename MACA-style lat/lon dims if needed
        if "lat" in da.dims and "lon" in da.dims:
            da = da.rio.set_spatial_dims(x_dim="lon", y_dim="lat")

        # Fix 0–360 longitude to -180–180 if needed
        if "lon" in da.coords:
            lon_vals = da["lon"].values
            if lon_vals.max() > 180:
                da = da.assign_coords(
                    lon=((da.lon + 180) % 360) - 180
                ).sortby("lon")

        # Write CRS
        da = da.rio.write_crs("EPSG:4326")

        # Clip if requested
        if clip_bounds is not None:
            da = da.rio.clip_box(*clip_bounds)

        out_name = os.path.basename(fp).replace(".nc", ".tif")
        out_fp = os.path.join(out_dir, out_name)

        da.rio.to_raster(out_fp)
        tif_files.append(out_fp)

        ds.close()
        print(f"Saved {out_fp}")

    return tif_files

In [ ]:
gun_tifs = convert_maca_nc_to_tif(
    nc_dir=maca_dir,
    out_dir=os.path.join(maca_dir, "tifs_gunnison"),
    variable="precipitation",
    summary_method="mean",
    clip_bounds=gun_gdf.total_bounds
)

In [ ]:
   input_rasters = [
    f"{data_dir}/maca-dir/tifs_gunnison*.tif"]
    data_dir = 

In [ ]:
# Harmonized the layers
harmonized_rasters = harmonize_raster_layers_gun(
    reference_raster_gun = f"{data_dir}/site_aspen_CO/soil pH/rasters/Gunnison_soil_ph.tif",
    
 
)

## Create Fuzzy Logic Model

In [ ]:
def fuzzy_score(x, min_val, optimal, max_val):

    """
    Collect the optimal values for each habitat suitability metric and output the suitability score of the grid point

    Args: 
    x (str): Habitat suitability variable of interest
    min_val (int): Lower end of optimal values
    optimal (int): Optimal value for that suitability variable
    max_val (int): Upper end of optimal values

    """ 
    
    return np.maximum(
        np.minimum((x - min_val) / (optimal - min_val),
                   (max_val - x) / (max_val - optimal)), 0
    )

In [ ]:
fuzzy_optimal_gun = {
    "ph": (6.0, 6.75, 7.5),
    "elevation": (2400, 2700, 3000),
    "northness": (.3, 1.0, .3), # Aspect as degree did not multiply well
    "om": (4, 6, 8)
    
}

In [ ]:
layers_gun = {
    "ph": reproj_da_list_gun[0],
    "elevation": reproj_da_list_gun[1],
    "northness": reproj_da_list_gun[2],
    "om": reproj_da_list_gun[3]
    
}

In [ ]:
aligned_layers_gun = []

for da in reproj_da_list_gun:
    da2 = da.squeeze()
    aligned_layers_gun.append(da2)

In [ ]:
template = aligned_layers_gun[0]

for i, da in enumerate(aligned_layers):
    print(da.shape, da.x.equals(template.x), da.y.equals(template.y))

In [ ]:
fuzzy_layers_gun = {
    name: fuzzy_score(da, *fuzzy_optimal_gun[name])
    for name, da in layers_gun.items()
}

In [ ]:
hsi_gun = None

for da in fuzzy_layers_gun.values():
    if hsi_gun is None:
        hsi_gun = da.copy()
    else:
        hsi_gun = hsi_gun * da

In [ ]:
if not fuzzy_layers_gun:
    raise ValueError("fuzzy_layers_gun is empty")

hsi_gun = list(fuzzy_layers_gun.values())[0].fillna(0).copy()

for da in list(fuzzy_layers_gun.values())[1:]:
    hsi_gun = hsi_gun * da.fillna(0)

In [ ]:
print(fuzzy_layers_gun)

In [ ]:
print(hsi_gun)

In [ ]:
ph_suit_gun = fuzzy_score(reproj_da_list_gun[0], 0.3, 1.0, 0.3)
elev_suit_gun = fuzzy_score(reproj_da_list_gun[1], 2400, 2700, 3000)
north_suit_gun = fuzzy_score(reproj_da_list_gun[2], 0.3, 1.0, 0.3)
om_suit_gun = fuzzy_score(reproj_da_list_gun[3], 4, 6, 8)


In [ ]:
fuzzy_optimal_gun = {
    "ph": (6.0, 6.75, 7.5),
    "elevation": (2400, 2700, 3000),
    "northness": (.3, 1.0, .3), # Aspect as degree did not multiply well
    "om": (4, 6, 8)

    
    
}

In [ ]:
north_suit.plot(cmap="YlGn", vmin=0, vmax=1)

In [ ]:
# Create subplots for Fishlake National Forest
fig, axes = plt.subplots(1, len(reproj_da_list_gun),
                         figsize = (5*len(reproj_da_list_gun), 5))

ph_suit_gun.plot(cmap="YlGn", vmin=0, vmax=1)
elev_suit_gun.plot(cmap="YlGn", vmin=0, vmax=1)
north_suit_gun.plot(cmap="YlGn", vmin=0, vmax=1)
om_suit_gun.plot(cmap="YlGn", vmin=0, vmax=1)
gun_gdf.plot(ax = ax, facecolor = 'none', edgecolor = 'white', linewidth = 1)

ax.set_aspect("equal")
ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
np.isnan(gun_aspect.values).sum()

In [ ]:
hsi_gun.plot(cmap="YlGn", vmin=0, vmax=1)

In [ ]:
hsi_gun.plot(robust=True, cmap="YlGn")

In [ ]:
gun_layers = {
    "ph": gun_soil_ph_da,
    "om": gun_soil_om_da,
    "elev": gun_srtm_da,
    "aspect": gun_aspect
}

fuzzy_params = {
    "ph": (6.0, 6.75, 8.0),
    "om": (2400, 2700, 3000),
    "elev": (2200, 2800, 3500),
    "aspect":(315, 360, 45)
}

gun_fuzzy_layers = {}

for name, da in gun_layers.items():
    params = fuzzy_params[name]
    gun_fuzzy_layers[name] = fuzzy_score(da, *params)

hsi_gun = list(gun_fuzzy_layers.values())[0].copy()

for da in list(gun_fuzzy_layers.values())[1:]:
    hsi_gun = hsi_gun * da  

for name, da in gun_fuzzy_layers.items():
    print(f"\n{name}")
    print("shape:", da.shape)
    print("crs:", da.rio.crs)
    print("bounds:", da.rio.bounds())
    print("finite:", np.isfinite(da.values).sum())

In [ ]:
ph_suit_gun = fuzzy_score(gun_ph_da, 6.0, 6.75, 7.5)
elev_suit_gun = fuzzy_score(gun_srtm_da, 2400, 2700, 3000)
om_suit_gun = fuzzy_score(gun_om_da, 4, 6, 8)
aspect_suit_gun = fuzzy_score(gun_aspect, 315, 360, 45)


In [ ]:
print(gun_fuzzy_layers)

In [ ]:
print(reproj_da_list_gun[0])
print(reproj_da_list_gun[1])
print(reproj_da_list_gun[2])
print(reproj_da_list_gun[3])

In [ ]:
plt.figure(figsize=(8,6))

hsi_gun.plot(
    cmap="YlGn",
    vmin=0,
    vmax=1
)

plt.title("Habitat Suitability Index")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()

**Aspens have an interesting journey across the US especially in the National Forests of the west like Gunnison National Forest in Colorado and Fishlake National Forest in Utah.**